<a href="https://colab.research.google.com/github/ritaa-ml/Freigh-rate-prediction/blob/main/code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import os

In [ ]:
from google.colab import files
uploaded = files.upload()   # in the picker: open Documents, select train-test.csv, validation.csv, december-chart-inputs.csv
print(list(uploaded.keys()))

In [ ]:
import os, json
import numpy as np
import pandas as pd

TARGET = "posted_rate"
DATE_FMT = "%m/%d/%Y"
EQUIP = {"Dry Van": 0, "Reefer": 1, "Flatbed": 2}
CAT_COLS = ["equipment", "pickup", "delivery"]
SHORT_HAUL_MILES = 600

def find(*names):
    for n in names:
        for folder in ["", "data/"]:
            if os.path.exists(folder + n):
                return folder + n
    raise FileNotFoundError(f"Upload one of: {names}")

def load(path):
    df = pd.read_csv(path)
    df["date"] = pd.to_datetime(df["date"], format=DATE_FMT, errors="coerce")
    return df

raw_tr  = load(find("train-test.csv", "train_test.csv"))
raw_val = load(find("validation.csv"))
raw_dec = load(find("december-chart-inputs.csv", "december_chart_inputs.csv"))
print(raw_tr.shape, raw_val.shape, raw_dec.shape)   # expect (48000, 14) (12000, 13) (31, 7)

In [ ]:
def quality_report(df, name):
    rep = {
        "dataset": name, "rows": int(len(df)),
        "date_range": [str(df["date"].min().date()), str(df["date"].max().date())],
        "unparseable_dates": int(df["date"].isna().sum()),
        "missing_values": {k: int(v) for k, v in df.isna().sum().items() if v > 0},
        "duplicate_load_id": int(df["load_id"].duplicated().sum()),
        "negative_weight": int((df["weight"] < 0).sum()),
        "weight_exactly_-47500": int((df["weight"] == -47500).sum()),
        "nonpositive_distance": int((df["distance"] <= 0).sum()),
        "unknown_equipment": int((~df["equipment"].isin(EQUIP)).sum()),
    }
    if TARGET in df:
        rep["nonpositive_target"] = int((df[TARGET] <= 0).sum())
        ratio = df[TARGET] / (df["quote_signal"] * df["distance"])
        rep["rate_vs_quote_x_distance_pctiles"] = ratio.quantile([.001, .01, .5, .99, .999]).round(3).to_dict()
    return rep

print(json.dumps([quality_report(raw_tr, "train"), quality_report(raw_val, "validation")], indent=2))

[
  {
    "dataset": "train",
    "rows": 48000,
    "date_range": [
      "2025-01-01",
      "2025-10-31"
    ],
    "unparseable_dates": 0,
    "missing_values": {
      "weight": 300,
      "market_index": 374
    },
    "duplicate_load_id": 0,
    "negative_weight": 292,
    "weight_exactly_-47500": 13,
    "nonpositive_distance": 0,
    "unknown_equipment": 0,
    "nonpositive_target": 0,
    "rate_vs_quote_x_distance_pctiles": {
      "0.001": 0.207,
      "0.01": 0.732,
      "0.5": 1.004,
      "0.99": 2.546,
      "0.999": 4.975
    }
  },
  {
    "dataset": "validation",
    "rows": 12000,
    "date_range": [
      "2025-11-01",
      "2025-12-31"
    ],
    "unparseable_dates": 0,
    "missing_values": {
      "weight": 165,
      "market_index": 249
    },
    "duplicate_load_id": 0,
    "negative_weight": 145,
    "weight_exactly_-47500": 4,
    "nonpositive_distance": 0,
    "unknown_equipment": 0
  }
]


In [ ]:
def clean(df, ref_weight_median):
    """Never drops rows, so every validation load still gets a prediction."""
    df = df.copy()
    # (a) negative weights = sign errors (includes the -47500 sentinel) -> absolute value + flag
    df["weight_was_negative"] = (df["weight"] < 0).astype(int)
    df["weight"] = df["weight"].abs()
    # (b) missing weight -> median weight of the same equipment type (from TRAIN) + flag
    df["weight_was_missing"] = df["weight"].isna().astype(int)
    df["weight"] = df["weight"].fillna(df["equipment"].map(ref_weight_median))
    # (c) missing market_index -> median market_index of the same date
    df["market_index_was_missing"] = df["market_index"].isna().astype(int)
    df["market_index"] = df["market_index"].fillna(
        df.groupby("date")["market_index"].transform("median"))
    return df

# medians come from TRAIN only and are reused for validation
wmed = raw_tr.assign(weight=raw_tr["weight"].abs()).groupby("equipment")["weight"].median()

tr_clean  = clean(raw_tr,  wmed)
val_clean = clean(raw_val, wmed)

print(tr_clean[["weight", "market_index"]].isna().sum())   # should be all 0
print(val_clean[["weight", "market_index"]].isna().sum())  # should be all 0

weight          0
market_index    0
dtype: int64
weight          0
market_index    0
dtype: int64


In [ ]:
def add_day_aggregates(df):
    """Date-level market context from that day's loads (no labels used).
    The daily average quote_signal reveals the 'surge' regime."""
    df = df.copy()
    g = df.groupby("date")
    df["day_qs"] = g["quote_signal"].transform("mean")
    df["day_mi"] = g["market_index"].transform("mean")
    short = df["quote_signal"].where(df["distance"] < SHORT_HAUL_MILES)
    df["day_short_qs"] = short.groupby(df["date"]).transform("mean")
    return df

def build_lookups(frames):
    cities = sorted(set().union(*[set(f["pickup"]) | set(f["delivery"]) for f in frames]))
    return {"city": {c: i for i, c in enumerate(cities)}}

def make_features(df, lk):
    X = pd.DataFrame(index=df.index)
    X["log_distance"]      = np.log(df["distance"])
    X["weight"]            = df["weight"]
    X["equipment"]         = df["equipment"].map(EQUIP)
    X["pickup"]            = df["pickup"].map(lk["city"])
    X["delivery"]          = df["delivery"].map(lk["city"])
    X["market_index"]      = df["market_index"]
    X["log_quote_signal"]  = np.log(df["quote_signal"])
    X["log_implied_rate"]  = X["log_quote_signal"] + X["log_distance"]   # log(quote_signal * distance)
    X["dow"]               = df["date"].dt.dayofweek
    X["day_qs"]            = df["day_qs"]
    X["day_mi"]            = df["day_mi"]
    X["qs_vs_day"]         = df["quote_signal"] / df["day_qs"]
    X["day_short_qs"]      = df["day_short_qs"]
    return X

train = add_day_aggregates(tr_clean).sort_values("date").reset_index(drop=True)
val   = add_day_aggregates(val_clean)
lk    = build_lookups([train, val, raw_dec])

X  = make_features(train, lk)
Xv = make_features(val, lk)

# training target: log(rate), with the rate/(quote*distance) ratio clipped at the 0.1 / 99.9 percentiles (TRAIN only)
log_ratio = np.log(train[TARGET]) - X["log_implied_rate"]
lo, hi = log_ratio.quantile([0.001, 0.999])
y_log = X["log_implied_rate"] + log_ratio.clip(lo, hi)
y = train[TARGET].values

print(X.shape, Xv.shape)
X.head()

(48000, 13) (12000, 13)


,log_distance,weight,equipment,pickup,delivery,market_index,log_quote_signal,log_implied_rate,dow,day_qs,day_mi,qs_vs_day,day_short_qs
0,5.614222,30658.0,0,58,7,0.95684,0.873780,6.488002,2,2.052912,0.961233,1.167098,2.305142
1,6.539152,47500.0,0,47,45,0.95310,0.773754,7.312907,2,2.052912,0.961233,1.056007,2.305142
2,7.001246,29071.0,0,29,43,0.96395,0.597885,7.599131,2,2.052912,0.961233,0.885703,2.305142
3,6.528104,25756.0,0,29,12,0.92209,0.643799,7.171903,2,2.052912,0.961233,0.927317,2.305142
4,6.793018,21332.0,0,40,57,0.97336,0.581981,7.374999,2,2.052912,0.961233,0.871727,2.305142


In [ ]:
daily = val.groupby("date").agg(day_qs=("quote_signal", "mean"),
                                day_mi=("market_index", "mean"),
                                day_short_qs=("day_short_qs", "first")).reset_index()
dec = raw_dec.merge(daily, on="date", how="left")
assert dec["day_qs"].notna().all(), "December dates missing from validation.csv"

dec["quote_signal"] = dec["day_qs"]
dec["market_index"] = dec["day_mi"]
dec["weight"] = dec["weight"].abs()
Xd = make_features(dec, lk)
print(Xd.shape)   # (31, 13)

(31, 13)


In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def new_model(columns):
    return HistGradientBoostingRegressor(
        loss="absolute_error",          # robust to the heavy right tail of rates
        learning_rate=0.05, max_iter=300, max_leaf_nodes=31, min_samples_leaf=40,
        l2_regularization=1.0, random_state=42,
        categorical_features=[c in CAT_COLS for c in columns])

def metrics(y_true, p):
    return {"MAE": mean_absolute_error(y_true, p),
            "RMSE": mean_squared_error(y_true, p) ** 0.5,
            "MAPE_pct": float(np.mean(np.abs(y_true - p) / y_true) * 100),
            "R2": r2_score(y_true, p)}

# Expanding window: train on all earlier months, test on the next month (never random)
month = train["date"].dt.month.values
rows = []
for m in range(5, 11):                                   # test months May..Oct
    tr_i, te_i = np.where(month < m)[0], np.where(month == m)[0]
    model = new_model(X.columns).fit(X.iloc[tr_i], y_log.iloc[tr_i])
    pred = np.exp(model.predict(X.iloc[te_i]))
    # baseline: quote_signal * distance * median historical ratio
    k = np.log(train[TARGET].iloc[tr_i] / np.exp(X["log_implied_rate"].iloc[tr_i])).median()
    base = np.exp(X["log_implied_rate"].iloc[te_i] + k).values
    rows.append({"test_month": m, "model": "GBM", **metrics(y[te_i], pred)})
    rows.append({"test_month": m, "model": "baseline_quote_x_distance", **metrics(y[te_i], base)})

cv = pd.DataFrame(rows)
print(cv.groupby("model")[["MAE", "RMSE", "MAPE_pct", "R2"]].mean().round(3))
cv.round(2)

                               MAE     RMSE  MAPE_pct     R2
model                                                       
GBM                        111.833  630.870     4.800  0.828
baseline_quote_x_distance  284.927  723.534    13.852  0.771


,test_month,model,MAE,RMSE,MAPE_pct,R2
0,5,GBM,92.52,557.11,4.45,0.86
1,5,baseline_quote_x_distance,403.46,731.04,20.98,0.76
2,6,GBM,122.77,703.88,4.42,0.81
3,6,baseline_quote_x_distance,83.45,694.59,2.84,0.81
4,7,GBM,97.64,629.49,4.00,0.83
5,7,baseline_quote_x_distance,416.03,793.09,20.64,0.72
6,8,GBM,157.20,630.21,6.95,0.82
7,8,baseline_quote_x_distance,318.54,713.72,14.73,0.77
8,9,GBM,94.16,614.46,3.95,0.84
9,9,baseline_quote_x_distance,73.05,608.84,2.95,0.84


In [ ]:
import glob

final = new_model(X.columns).fit(X, y_log)               # train on ALL labeled data

pred_val = np.exp(final.predict(Xv)).round(2)
sub = pd.DataFrame({"load_id": val["load_id"].values, "predicted_rate": pred_val})

# keep the template's row order, if you have the template file
tmpl = glob.glob("**/*template*.csv", recursive=True)
if tmpl:
    sub = pd.read_csv(tmpl[0])[["load_id"]].merge(sub, on="load_id", how="left")

assert len(sub) == 12_000 and sub["load_id"].is_unique
assert sub["predicted_rate"].notna().all() and (sub["predicted_rate"] > 0).all()
sub.to_csv("validation_predictions.csv", index=False)
print(sub.shape); sub.head()

(12000, 2)


,load_id,predicted_rate
0,TE-000001,839.78
1,TE-000002,4784.07
2,TE-000003,5279.42
3,TE-000004,4218.01
4,TE-000005,1786.07


In [ ]:
pred_dec = np.exp(final.predict(Xd)).round(2)
dec_out = raw_dec.copy()                                 # keeps the original 7 columns and order
dec_out["predicted_rate"] = pred_dec
dec_out["date"] = dec_out["date"].dt.strftime("%Y-%m-%d")
os.makedirs("outputs", exist_ok=True)
dec_out.to_csv("outputs/december_predictions.csv", index=False)
dec_out

,pickup,delivery,distance,equipment,weight,date,predicted_rate
0,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-01,804.64
1,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-02,808.28
2,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-03,813.64
3,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-04,803.13
4,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-05,813.79
5,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-06,804.21
6,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-07,803.65
7,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-08,802.56
8,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-09,811.06
9,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-10,814.46


In [ ]:
from google.colab import files
files.download("validation_predictions.csv")
files.download("outputs/december_predictions.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from __future__ import annotations

import argparse
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


EXPECTED_ROWS = 12_000
EXPECTED_IDS = {f"TE-{index:06d}" for index in range(1, EXPECTED_ROWS + 1)}
DECEMBER_DATES = pd.date_range("2025-12-01", "2025-12-31", freq="D")
FIXED_PICKUP = "Lexington"
FIXED_DELIVERY = "Fort Wayne"
FIXED_DISTANCE = 360.0
FIXED_EQUIPMENT = "Dry Van"
FIXED_WEIGHT = 32_000.0


def fail(message: str) -> None:
    raise SystemExit(f"ERROR: {message}")


def read_csv(path: Path, label: str) -> pd.DataFrame:
    if not path.is_file():
        fail(f"{label} file not found: {path}")
    try:
        return pd.read_csv(path)
    except Exception as exc:
        fail(f"could not read {label}: {exc}")


def numeric_series(frame: pd.DataFrame, column: str, label: str) -> pd.Series:
    values = pd.to_numeric(frame[column], errors="coerce")
    if values.isna().any() or not np.isfinite(values).all():
        fail(f"{label} contains invalid {column} values")
    return values.astype(float)


def validate_predictions(predictions: pd.DataFrame) -> None:
    if list(predictions.columns) != ["load_id", "predicted_rate"]:
        fail("predictions must contain exactly two columns in this order: load_id,predicted_rate")
    if len(predictions) != EXPECTED_ROWS:
        fail(f"predictions must contain exactly {EXPECTED_ROWS:,} rows")
    if predictions["load_id"].isna().any() or predictions["load_id"].duplicated().any():
        fail("predictions contains missing or duplicate load_id values")

    submitted_ids = set(predictions["load_id"].astype(str))
    missing = EXPECTED_IDS - submitted_ids
    extra = submitted_ids - EXPECTED_IDS
    if missing or extra:
        fail(
            "prediction IDs do not match the validation set "
            f"(missing={len(missing)}, extra={len(extra)})"
        )

    predicted_rate = numeric_series(predictions, "predicted_rate", "predictions")
    if (predicted_rate <= 0).any():
        fail("predictions contains non-positive predicted_rate values")


def validate_december(frame: pd.DataFrame) -> pd.DataFrame:
    columns = ["pickup", "delivery", "distance", "equipment", "weight", "date", "predicted_rate"]
    if list(frame.columns) != columns:
        fail("December predictions must keep the original seven columns and column order")

    result = frame.copy()
    result["date"] = pd.to_datetime(result["date"], errors="coerce")
    if result["date"].isna().any():
        fail("December predictions contains invalid dates")
    result["distance"] = numeric_series(result, "distance", "December predictions")
    result["weight"] = numeric_series(result, "weight", "December predictions")
    result["predicted_rate"] = numeric_series(result, "predicted_rate", "December predictions")

    if result["date"].duplicated().any():
        fail("December predictions contains duplicate dates")
    if len(result) != 31 or set(result["date"]) != set(DECEMBER_DATES):
        fail("December predictions must contain one row for every day from 2025-12-01 to 2025-12-31")
    if not result["pickup"].eq(FIXED_PICKUP).all():
        fail(f"December pickup must be {FIXED_PICKUP} for all rows")
    if not result["delivery"].eq(FIXED_DELIVERY).all():
        fail(f"December delivery must be {FIXED_DELIVERY} for all rows")
    if not np.isclose(result["distance"], FIXED_DISTANCE).all():
        fail(f"December distance must be {FIXED_DISTANCE:g} for all rows")
    if not result["equipment"].eq(FIXED_EQUIPMENT).all():
        fail(f"December equipment must be {FIXED_EQUIPMENT} for all rows")
    if not np.isclose(result["weight"], FIXED_WEIGHT).all():
        fail(f"December weight must be {FIXED_WEIGHT:g} for all rows")
    if (result["predicted_rate"] <= 0).any():
        fail("December predicted_rate values must be positive")
    return result.sort_values("date")


def save_december_chart(december: pd.DataFrame, output: Path) -> None:
    figure, axis = plt.subplots(figsize=(10.8, 4.8), dpi=180)
    color = "#064A56"
    axis.plot(
        december["date"],
        december["predicted_rate"],
        color=color,
        linewidth=2.6,
        marker="o",
        markersize=3.2,
    )
    floor = float(december["predicted_rate"].min())
    axis.fill_between(
        december["date"],
        december["predicted_rate"],
        floor - max(10.0, floor * 0.02),
        color=color,
        alpha=0.08,
    )
    axis.set_title("Candidate: December 2025 Predicted Load Rate", loc="left", fontsize=15, fontweight="bold", pad=12)
    axis.set_ylabel("Predicted rate ($)")
    axis.grid(axis="y", color="#D9E2E4", linewidth=0.8)
    axis.spines[["top", "right"]].set_visible(False)
    axis.spines[["left", "bottom"]].set_color("#9DAFB3")
    axis.tick_params(axis="x", rotation=35)
    axis.text(
        0,
        -0.40,
        "Fixed inputs: Lexington to Fort Wayne | 360 miles | Dry Van | 32,000 lb | only date changes",
        transform=axis.transAxes,
        fontsize=9.5,
        color="#455A60",
    )
    figure.tight_layout(rect=(0, 0.12, 1, 1))
    figure.savefig(output, bbox_inches="tight")
    plt.close(figure)


def main() -> None:
    parser = argparse.ArgumentParser(
        description="Validate candidate output files and generate the fixed December chart."
    )
    parser.add_argument("--predictions", required=True, help="CSV with load_id,predicted_rate")
    parser.add_argument(
        "--december-predictions",
        required=True,
        help="Completed data/december_chart_inputs.csv",
    )
    parser.add_argument("--output-dir", default="scorer_results")
    args = parser.parse_args()

    validate_predictions(read_csv(Path(args.predictions), "predictions"))
    december = validate_december(read_csv(Path(args.december_predictions), "December predictions"))
    output = Path(args.output_dir)
    output.mkdir(parents=True, exist_ok=True)
    chart = output / "candidate_december.png"
    save_december_chart(december, chart)

    print(f"Validated {EXPECTED_ROWS:,} final predictions.")
    print("Validated 31 fixed December predictions.")
    print(f"Created chart: {chart}")
    print("Final validation metrics are calculated by Spotter after submission.")


if __name__ == "__main__":
    main()

usage: colab_kernel_launcher.py [-h] --predictions PREDICTIONS
                                --december-predictions DECEMBER_PREDICTIONS
                                [--output-dir OUTPUT_DIR]
colab_kernel_launcher.py: error: the following arguments are required: --predictions, --december-predictions
ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/lib/python3.13/argparse.py", line 1965, in _parse_known_args2
    namespace, args = self._parse_known_args(args, namespace, intermixed)
                      ~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.13/argparse.py", line 2262, in _parse_known_args
    raise ArgumentError(None, _('the following arguments are required: %s') %
               ', '.join(required_actions))
argparse.ArgumentError: the following arguments are required: --predictions, --december-predictions

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_1155/3601159154.py", line 163, in <cell line: 0>
    main()
    ~~~~^^
  File "/tmp/ipykernel_1155/36011

TypeError: object of type 'NoneType' has no len()

In [ ]:
from __future__ import annotations

import argparse
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


EXPECTED_ROWS = 12_000
EXPECTED_IDS = {f"TE-{index:06d}" for index in range(1, EXPECTED_ROWS + 1)}
DECEMBER_DATES = pd.date_range("2025-12-01", "2025-12-31", freq="D")
FIXED_PICKUP = "Lexington"
FIXED_DELIVERY = "Fort Wayne"
FIXED_DISTANCE = 360.0
FIXED_EQUIPMENT = "Dry Van"
FIXED_WEIGHT = 32_000.0


def fail(message: str) -> None:
    raise SystemExit(f"ERROR: {message}")


def read_csv(path: Path, label: str) -> pd.DataFrame:
    if not path.is_file():
        fail(f"{label} file not found: {path}")
    try:
        return pd.read_csv(path)
    except Exception as exc:
        fail(f"could not read {label}: {exc}")


def numeric_series(frame: pd.DataFrame, column: str, label: str) -> pd.Series:
    values = pd.to_numeric(frame[column], errors="coerce")
    if values.isna().any() or not np.isfinite(values).all():
        fail(f"{label} contains invalid {column} values")
    return values.astype(float)


def validate_predictions(predictions: pd.DataFrame) -> None:
    if list(predictions.columns) != ["load_id", "predicted_rate"]:
        fail("predictions must contain exactly two columns in this order: load_id,predicted_rate")
    if len(predictions) != EXPECTED_ROWS:
        fail(f"predictions must contain exactly {EXPECTED_ROWS:,} rows")
    if predictions["load_id"].isna().any() or predictions["load_id"].duplicated().any():
        fail("predictions contains missing or duplicate load_id values")

    submitted_ids = set(predictions["load_id"].astype(str))
    missing = EXPECTED_IDS - submitted_ids
    extra = submitted_ids - EXPECTED_IDS
    if missing or extra:
        fail(
            "prediction IDs do not match the validation set "
            f"(missing={len(missing)}, extra={len(extra)})"
        )

    predicted_rate = numeric_series(predictions, "predicted_rate", "predictions")
    if (predicted_rate <= 0).any():
        fail("predictions contains non-positive predicted_rate values")


def validate_december(frame: pd.DataFrame) -> pd.DataFrame:
    columns = ["pickup", "delivery", "distance", "equipment", "weight", "date", "predicted_rate"]
    if list(frame.columns) != columns:
        fail("December predictions must keep the original seven columns and column order")

    result = frame.copy()
    result["date"] = pd.to_datetime(result["date"], errors="coerce")
    if result["date"].isna().any():
        fail("December predictions contains invalid dates")
    result["distance"] = numeric_series(result, "distance", "December predictions")
    result["weight"] = numeric_series(result, "weight", "December predictions")
    result["predicted_rate"] = numeric_series(result, "predicted_rate", "December predictions")

    if result["date"].duplicated().any():
        fail("December predictions contains duplicate dates")
    if len(result) != 31 or set(result["date"]) != set(DECEMBER_DATES):
        fail("December predictions must contain one row for every day from 2025-12-01 to 2025-12-31")
    if not result["pickup"].eq(FIXED_PICKUP).all():
        fail(f"December pickup must be {FIXED_PICKUP} for all rows")
    if not result["delivery"].eq(FIXED_DELIVERY).all():
        fail(f"December delivery must be {FIXED_DELIVERY} for all rows")
    if not np.isclose(result["distance"], FIXED_DISTANCE).all():
        fail(f"December distance must be {FIXED_DISTANCE:g} for all rows")
    if not result["equipment"].eq(FIXED_EQUIPMENT).all():
        fail(f"December equipment must be {FIXED_EQUIPMENT} for all rows")
    if not np.isclose(result["weight"], FIXED_WEIGHT).all():
        fail(f"December weight must be {FIXED_WEIGHT:g} for all rows")
    if (result["predicted_rate"] <= 0).any():
        fail("December predicted_rate values must be positive")
    return result.sort_values("date")


def save_december_chart(december: pd.DataFrame, output: Path) -> None:
    figure, axis = plt.subplots(figsize=(10.8, 4.8), dpi=180)
    color = "#064A56"
    axis.plot(
        december["date"],
        december["predicted_rate"],
        color=color,
        linewidth=2.6,
        marker="o",
        markersize=3.2,
    )
    floor = float(december["predicted_rate"].min())
    axis.fill_between(
        december["date"],
        december["predicted_rate"],
        floor - max(10.0, floor * 0.02),
        color=color,
        alpha=0.08,
    )
    axis.set_title("Candidate: December 2025 Predicted Load Rate", loc="left", fontsize=15, fontweight="bold", pad=12)
    axis.set_ylabel("Predicted rate ($)")
    axis.grid(axis="y", color="#D9E2E4", linewidth=0.8)
    axis.spines[["top", "right"]].set_visible(False)
    axis.spines[["left", "bottom"]].set_color("#9DAFB3")
    axis.tick_params(axis="x", rotation=35)
    axis.text(
        0,
        -0.40,
        "Fixed inputs: Lexington to Fort Wayne | 360 miles | Dry Van | 32,000 lb | only date changes",
        transform=axis.transAxes,
        fontsize=9.5,
        color="#455A60",
    )
    figure.tight_layout(rect=(0, 0.12, 1, 1))
    figure.savefig(output, bbox_inches="tight")
    plt.close(figure)


def main() -> None:
    parser = argparse.ArgumentParser(
        description="Validate candidate output files and generate the fixed December chart."
    )
    parser.add_argument("--predictions", required=True, help="CSV with load_id,predicted_rate")
    parser.add_argument(
        "--december-predictions",
        required=True,
        help="Completed data/december_chart_inputs.csv",
    )
    parser.add_argument("--output-dir", default="scorer_results")
    args = parser.parse_args()

    validate_predictions(read_csv(Path(args.predictions), "predictions"))
    december = validate_december(read_csv(Path(args.december_predictions), "December predictions"))
    output = Path(args.output_dir)
    output.mkdir(parents=True, exist_ok=True)
    chart = output / "candidate_december.png"
    save_december_chart(december, chart)

    print(f"Validated {EXPECTED_ROWS:,} final predictions.")
    print("Validated 31 fixed December predictions.")
    print(f"Created chart: {chart}")
    print("Final validation metrics are calculated by Spotter after submission.")


if __name__ == "__main__":
    main()

# New Section